In [5]:
from xgboost import XGBRegressor
import requests
import pandas as pd
import sklearn
import requests
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

#CURRENT

ERCOT_primary_key = '1b766749760c4fdcbc11936fbf9853eb'
day_ahead = "2024-09-01"
end_date = "2026-09-01"
auth_url = (
"https://ercotb2c.b2clogin.com/"
"ercotb2c.onmicrosoft.com/"
"B2C_1_PUBAPI-ROPC-FLOW/oauth2/v2.0/token")

payload = {"username": "tylernorman1212@gmail.com", "password": "YUZ4.f$j6J5RxY@",
           "grant_type": "password","scope": "openid fec253ea-0d06-4272-a5e6-b478baeecd70 offline_access",
           "client_id": "fec253ea-0d06-4272-a5e6-b478baeecd70","response_type": "id_token"}

token_response = requests.post(auth_url, data=payload)
print(token_response.status_code)
token_json = token_response.json()
access_token = token_json["access_token"]

ERCOTheaders = {"Authorization": f"Bearer {access_token}", "Ocp-Apim-Subscription-Key": ERCOT_primary_key}
ERCOTparams = {"settlementPoint": "HB_HOUSTON", "deliveryDateFrom": day_ahead}
ERCOTLparams = { "deliveryDateFrom": day_ahead}
DAM_Settlement_Point_Prices_url = "https://api.ercot.com/api/public-reports/np4-190-cd/dam_stlmnt_pnt_prices"
response = requests.get(DAM_Settlement_Point_Prices_url, headers=ERCOTheaders,params=ERCOTparams)

print(response.status_code)
ERCOT_data = response.json()
ERCOT_DAM_df = pd.DataFrame(ERCOT_data['data'])
ERCOT_DAM_df = ERCOT_DAM_df.rename(columns={0: "deliveryDate", 1: "hourEnding", 3: "settlementPointPrice"})


ERCOT_DAM_df["hour"] = (ERCOT_DAM_df["hourEnding"].str.split(":").str[0].astype(int))
ERCOT_DAM_df["timestamp"] = (pd.to_datetime(ERCOT_DAM_df["deliveryDate"])+ pd.to_timedelta(ERCOT_DAM_df["hour"] - 1, unit="h"))
ERCOT_DAM_df["hour"] = ERCOT_DAM_df["timestamp"].dt.hour
ERCOT_DAM_df = ERCOT_DAM_df.sort_values("timestamp")
ERCOT_DAM_df["dayofweek"] = ERCOT_DAM_df["timestamp"].dt.dayofweek
ERCOT_DAM_df["month"] = ERCOT_DAM_df["timestamp"].dt.month
ERCOT_DAM_df["weekend"] = ERCOT_DAM_df["dayofweek"].isin([5,6]).astype(int)
ERCOT_DAM_df["lag_24"] = ERCOT_DAM_df["settlementPointPrice"].shift(24)
ERCOT_DAM_df["lag_48"] = ERCOT_DAM_df["settlementPointPrice"].shift(48)
ERCOT_DAM_df["lag_168"] = ERCOT_DAM_df["settlementPointPrice"].shift(168)
ERCOT_DAM_df["roll24"] = (ERCOT_DAM_df["settlementPointPrice"].rolling(24).mean())
ERCOT_DAM_df["roll168"] = (ERCOT_DAM_df["settlementPointPrice"].rolling(168).mean())

ERCOT_DAM_df.to_excel(
    "ERCOT_HB_HOUSTON_DAM.xlsx",
    index=False
)

#weather data 
url = "https://archive-api.open-meteo.com/v1/archive"

params = {"latitude": 29.98, "longitude": -95.36, "start_date": day_ahead, "end_date": end_date, "hourly": ["temperature_2m", "relative_humidity_2m",  "precipitation","wind_speed_10m" ]}
r = requests.get(url, params=params)
data = r.json()

hist_weather_df = pd.DataFrame(data["hourly"])
hist_weather_df["timestamp"] = pd.to_datetime(hist_weather_df["time"])

merged_ercot_hist_df = ERCOT_DAM_df.merge(hist_weather_df, on = "timestamp", how = "left")


#Train the Model
features = ["hour","dayofweek","weekend","month", "lag_24","lag_48","lag_168","roll24", "roll168","temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m"]
train = merged_ercot_hist_df.dropna().sort_values("timestamp")
X = train[features]
y = train["settlementPointPrice"]
split_idx = int(len(train) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

model = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    objective="reg:squarederror",
    random_state=42
)

model.fit(X_train, y_train)
preds = model.predict(X_test)
r2 = r2_score(y_test, preds)
mae = mean_absolute_error(y_test, preds)
print(f"XGBoost MAE: {mae:.3f}")
naive_preds = X_test["lag_24"]
naive_mae = mean_absolute_error(y_test, naive_preds)
print(f"Naive MAE: {naive_mae:.3f}")
improvement = (naive_mae - mae) / naive_mae * 100
print(f"Improvement over naive: {improvement:.2f}%")
# Feature importance
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)
print("\nFeature Importances:")
print(importance)

200
200
XGBoost MAE: 7.095
Naive MAE: 8.664
Improvement over naive: 18.11%

Feature Importances:
lag_24                  0.623827
hour                    0.143208
roll24                  0.063437
wind_speed_10m          0.061898
temperature_2m          0.028017
lag_168                 0.022256
roll168                 0.015698
dayofweek               0.013904
lag_48                  0.012924
relative_humidity_2m    0.011119
precipitation           0.003714
month                   0.000000
weekend                 0.000000
dtype: float32


In [4]:
pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use update